# LTX-2.5 Distilled — idea_16nov26 Part A (Interview + Carnival + Bus Stop + Coach Warren)

14 clips total (random seeds):
- **Character Interview (`9:16`, 3 clips):** Younger man + older woman talking in a fancy restaurant
- **Narrated Carnival Couple (`9:16`, 2 chained clips):** Black man & Asian woman at a carnival at night, off-camera narrator
- **Cinematic Bus Stop (`16:9` x2 chained + `9:16` x2 chained = 4 clips):** Middle-aged man & pretty woman dialogue + walk-away punchline
- **Coach Warren (`9:16`, 5 slow-cadence clips):** Pickup lines vs situational conversation (`coach_warren_ref_9x16.jpg`, `0.75 guide`)


In [ ]:
#@title 1. Check Hardware & Capture System Baseline
import os, subprocess

gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'
], text=True).strip().splitlines()[0]
gpu_name, gpu_vram, gpu_drv = [x.strip() for x in gpu_line.split(',')]
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3)
print(f'=== MACHINE: {gpu_name} | VRAM: {gpu_vram} | Driver: {gpu_drv} | CPUs: {os.cpu_count()} | RAM: {ram_gb:.1f} GB ===')
!nvidia-smi
!df -h /content
!free -h


In [ ]:
#@title 2. Mount Google Drive (with /content/runs Fallback) & Create Run Folders
import datetime, os, time
from google.colab import drive

RUN_START_TS = time.time()
try:
    drive.mount('/content/drive')
    BASE_DRIVE_DIR = '/content/drive/MyDrive/ltx25_coach_runs'
except Exception as e:
    print('Drive mount failed, falling back to local /content/runs:', e)
    BASE_DRIVE_DIR = '/content/runs'

RUN_ID = f'run_16nov_{datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")}'
RUN_DIR = f'{BASE_DRIVE_DIR}/{RUN_ID}'
VIDEO_DIR = f'{RUN_DIR}/videos'
LOG_DIR = f'{RUN_DIR}/logs'
POLISHED_DIR = f'{RUN_DIR}/videos_polished'
ZIP_PATH = f'{BASE_DRIVE_DIR}/{RUN_ID}.zip'
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(POLISHED_DIR, exist_ok=True)
print('RUN_DIR =', RUN_DIR)


In [ ]:
#@title 3. Authenticate with Hugging Face (from Colab Secrets)
import os
from getpass import getpass
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not hf_token:
    hf_token = getpass('Enter your Hugging Face token (HF_TOKEN): ')
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('Hugging Face login succeeded.')

In [ ]:
#@title 4. Clone Lightricks/LTX-2 & Install Dependencies (+ natten)
%cd /content
!if [ ! -d LTX-2 ]; then git clone https://github.com/Lightricks/LTX-2.git; fi
%cd /content/LTX-2
!pip install -q uv
!uv python install 3.12
!uv sync --python 3.12
!uv sync --python 3.12 --inexact --package ltx-core --extra natten

In [ ]:
#@title 5. Download LTX-2.5 Distilled Weights (Skips Existing Files)
import os
from huggingface_hub import hf_hub_download

REPO_ID = 'Lightricks/LTX-2.5'
MODEL_DIR = '/content/models/ltx-2.5'
os.makedirs(MODEL_DIR, exist_ok=True)
FILES = [
    'diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    'vae/ltx-2.5-video-vae-bf16.safetensors',
    'vae/ltx-2.5-audio-vae-bf16.safetensors',
    'model_patches/ltx-2.5-duration-head-bf16.safetensors',
    'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
]
for rel_path in FILES:
    print(f'Ensuring {rel_path} ...')
    hf_hub_download(repo_id=REPO_ID, filename=rel_path, local_dir=MODEL_DIR)
!df -h /content


In [ ]:
#@title 6. Write Reference Frames (Coach Prime + Coach Warren 9:16), Save Env Info & Patch encode.py
import base64, os, shutil, subprocess

REF_B64 = '...'
WARREN_B64 = '...'
with open('/content/coach_ref.jpg', 'wb') as f:
    f.write(base64.b64decode(REF_B64))
with open('/content/coach_warren_ref.jpg', 'wb') as f:
    f.write(base64.b64decode(WARREN_B64))
# Create exact 9:16 center-crop of Coach Warren for vertical conditioning (no stretching)
subprocess.run([
    'ffmpeg', '-y', '-i', '/content/coach_warren_ref.jpg',
    '-vf', 'crop=ih*9/16:ih:(iw-ih*9/16)/2:0', '/content/coach_warren_ref_9x16.jpg'
], check=True, capture_output=True)
shutil.copy2('/content/coach_ref.jpg', f'{RUN_DIR}/coach_ref.jpg')
shutil.copy2('/content/coach_warren_ref.jpg', f'{RUN_DIR}/coach_warren_ref.jpg')
shutil.copy2('/content/coach_warren_ref_9x16.jpg', f'{RUN_DIR}/coach_warren_ref_9x16.jpg')
print('Wrote coach_ref.jpg, coach_warren_ref.jpg, and coach_warren_ref_9x16.jpg')

git_rev = subprocess.check_output(['git', '-C', '/content/LTX-2', 'rev-parse', 'HEAD'], text=True).strip()
gpu_csv = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,pci.bus_id,pcie.link.gen.current,pcie.link.width.current', '--format=csv,noheader'], text=True).strip()
with open(f'{LOG_DIR}/env_info.txt', 'w') as ef:
    ef.write(f'ltx2_git_rev={git_rev}\n')
    ef.write(f'gpu={gpu_csv}\n')

ENCODE_PY = '/content/LTX-2/packages/ltx-pipelines/src/ltx_pipelines/utils/media_io/encode.py'
shutil.copy2(ENCODE_PY, f'{LOG_DIR}/media_io_encode_orig.py')
with open(ENCODE_PY, 'r') as f:
    src = f.read()
patched = src.replace('_SDR_CRF = 19', '_SDR_CRF = 14').replace('_SDR_PRESET = "veryfast"', '_SDR_PRESET = "medium"')
if patched != src:
    with open(ENCODE_PY, 'w') as f:
        f.write(patched)
    shutil.copy2(ENCODE_PY, f'{LOG_DIR}/media_io_encode_patched.py')
    msg = 'Patched media_io/encode.py: _SDR_CRF=14, _SDR_PRESET="medium"'
else:
    msg = 'WARNING: _SDR_CRF = 19 / _SDR_PRESET = "veryfast" not found in media_io/encode.py'
print(msg)
with open(f'{LOG_DIR}/encode_patch.txt', 'w') as pf:
    pf.write(msg + '\n')


In [ ]:
#@title 7. Run Part A (14 Clips: Interview, Carnival, Bus Stop 16:9 & 9:16, Coach Warren 9:16)
import csv, json, os, random, shutil, subprocess, threading, time

def rand_seed():
    return random.randint(1, 2_000_000_000)

gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True
).strip().splitlines()[0]
GPU_NAME, gpu_mem_str = [x.strip() for x in gpu_info.split(',')]
GPU_VRAM_MIB = int(gpu_mem_str)
EXTRA_ARGS = []
if GPU_VRAM_MIB >= 70000:
    W16, H16 = 1536, 896   # True 16:9 (divisible by 64)
    W9, H9   = 896, 1536   # True 9:16 (divisible by 64)
else:
    W16, H16 = 1152, 640
    W9, H9   = 768, 1344
    EXTRA_ARGS = ['--quantization', 'fp8-cast']  # Halves 22B transformer VRAM so 241f fits on 40GB A100
print(f'=== MACHINE: {GPU_NAME} | VRAM: {GPU_VRAM_MIB} MiB ({GPU_VRAM_MIB/1024:.1f} GB) | 16:9={W16}x{H16} | 9:16={W9}x{H9} ===')

RUNPOD_A100_HOURLY_USD = 1.49
COLAB_A100_UNITS_PER_HR = 13.0
SUBPROC_ENV = {**os.environ, 'MPLBACKEND': 'Agg'}

WARREN_9X16_SCENE = (
    'Vertical 9:16 medium close-up portrait of the exact same distinguished 60s white man with silver hair and neatly groomed silver beard '
    'from the starting frame, sitting in a leather armchair inside an upscale modern golf clubhouse lounge, wearing a crisp white pique polo '
    'under a tailored slate-blue performance blazer, hands clasped. Soft directional window light, rich warm wood and blurred bar glassware '
    'in the background, no other people in the background, 85mm lens, f/1.4, Kodak Portra 400 aesthetic. '
    'He begins speaking smoothly on frame 1 at a slow, deliberate, unhurried, father-figure pace with natural pauses and subtle gestures. '
)
WARREN_AUDIO = (
    'Deep, warm, calm, reassuring baritone voice speaking slowly and deliberately, captured on an off-camera boom mic, '
    'no visible microphone, clean studio-grade speech, quiet upscale clubhouse room tone, no background music, no crowd noise.'
)

JOBS = [
    {
        'name': 'interview_01_restaurant', 'group': 'interview', 'character': 'interview_pair',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 24, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Vertical 9:16 cinematic medium two-shot inside a warmly lit upscale fancy restaurant at night with candlelight and softly blurred diners in the background. '
            'A charismatic younger man in his late 20s in a dark button-down shirt sits across from an elegant, sharp older woman in her 50s in a silk blouse. '
            'Both have natural synchronized mouth movements as they converse. First the younger man smiles and asks: "So what actually makes a man stand out to you?" '
            'Then the older woman leans forward, chuckles softly, and replies: "Honestly, a man who listens without trying too hard to impress." '
            'Natural voices, realistic close-mic restaurant dialogue, subtle clinking glasses in background.'
        ),
    },
    {
        'name': 'interview_02_restaurant', 'group': 'interview', 'character': 'interview_pair',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 25, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Vertical 9:16 cinematic medium two-shot in a refined upscale fine-dining restaurant with warm chandeliers and white tablecloths. '
            'A younger man in his mid-20s in a tailored jacket talks face-to-face with a poised older woman in her late 50s wearing pearl earrings. '
            'Synchronized lip-sync as they speak naturally. The older woman asks warmly: "Do you still get nervous walking up to someone new?" '
            'The younger man nods and answers honestly: "Every single time, I just learned to walk over anyway." '
            'Natural conversational voices, subtle quiet restaurant ambience.'
        ),
    },
    {
        'name': 'interview_03_restaurant', 'group': 'interview', 'character': 'interview_pair',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 24, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Vertical 9:16 cinematic angled two-shot inside a classy dimly lit restaurant booth. '
            'A younger man in his 20s and a confident older woman in her 50s share a lively, genuine conversation over dinner. '
            'Clear synchronized mouth movements for each speaker. The younger man says: "Most guys think they need the perfect opener." '
            'The older woman smiles knowingly and replies: "Half the time, just genuine curiosity is more than enough." '
            'Natural voices, realistic room acoustics, soft ambient restaurant murmur.'
        ),
    },
    {
        'name': 'carnival_01_walking_night', 'group': 'narrated_carnival', 'character': 'carnival_couple',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 29, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Vertical 9:16 cinematic 35mm night shot of a handsome Black man in his late 20s and a pretty Asian woman in her late 20s '
            'walking side by side through a glowing carnival midway at night with colorful Ferris wheel lights and neon booths behind them. '
            'They smile and enjoy the lights quietly; neither speaks and both keep their mouths closed the entire time. '
            'Over warm carnival ambience, an off-camera deep calm older male narrator voiceover (not lip-synced) says slowly: '
            '"Don\'t go to a fancy restaurant on the first date. Sitting face to face with someone you just met is not the best way to make them feel good."'
        ),
    },
    {
        'name': 'carnival_02_game_booth', 'group': 'narrated_carnival', 'character': 'carnival_couple',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 193, 'seed': rand_seed(),
        'word_count': 21, 'cond_mode': 'guide', 'cond_strength': 0.70, 'image_args': [], 'chain_from': 'carnival_01_walking_night',
        'prompt': (
            'Vertical 9:16 cinematic 35mm night shot continuing with the exact same Black man and Asian woman from the starting frame, '
            'now standing side by side at a bright carnival ring-toss game booth laughing together as he tosses a ring. '
            'They keep their mouths closed (no dialogue from them). Over soft carnival crowd ambience, an off-camera deep calm older male narrator voiceover says slowly: '
            '"The best thing you can do on the first date is something where you and your date will be side to side with activities."'
        ),
    },
    {
        'name': 'bus_stop_16x9_01_approach', 'group': 'cinematic_bus_stop_16x9', 'character': 'bus_stop_pair',
        'aspect': '16:9', 'width': W16, 'height': H16, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 17, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Cinematic 16:9 35mm golden-hour street scene with subtle tracking camera movement, warm acoustic underscore, and realistic city street ambience. '
            'A well-dressed middle-aged man in his 40s walks up with a friendly smile to a pretty woman waiting at a glass city bus stop. '
            'He says playfully: "Are you a keyboard? Because you\'re just my type!" '
            'The pretty woman laughs warmly and replies: "Nice try, but I have a boyfriend."'
        ),
    },
    {
        'name': 'bus_stop_16x9_02_punchline', 'group': 'cinematic_bus_stop_16x9', 'character': 'bus_stop_pair',
        'aspect': '16:9', 'width': W16, 'height': H16, 'num_frames': 169, 'seed': rand_seed(),
        'word_count': 4, 'cond_mode': 'guide', 'cond_strength': 0.70, 'image_args': [], 'chain_from': 'bus_stop_16x9_01_approach',
        'prompt': (
            'Cinematic 16:9 35mm continuation with the exact same middle-aged man and pretty woman at the city bus stop from the starting frame. '
            'The pretty woman smiles politely and walks away down the sidewalk. '
            'The middle-aged man watches her go, then turns to look directly at the camera with an amused grin and says: "At least she smiled." '
            'Light street ambience and subtle playful music sting.'
        ),
    },
    {
        'name': 'bus_stop_9x16_01_approach', 'group': 'cinematic_bus_stop_9x16', 'character': 'bus_stop_pair',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 241, 'seed': rand_seed(),
        'word_count': 17, 'cond_mode': 'none', 'cond_strength': 0.0, 'image_args': [], 'chain_from': None,
        'prompt': (
            'Vertical 9:16 cinematic 35mm golden-hour street scene with smooth camera movement, light indie underscore, and city bus-stop ambience. '
            'A well-dressed middle-aged man in his 40s walks up to a pretty woman waiting at a city bus stop. '
            'He smiles and says: "Are you a keyboard? Because you\'re just my type!" '
            'The pretty woman laughs and says: "Nice try, but I have a boyfriend."'
        ),
    },
    {
        'name': 'bus_stop_9x16_02_punchline', 'group': 'cinematic_bus_stop_9x16', 'character': 'bus_stop_pair',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': 169, 'seed': rand_seed(),
        'word_count': 4, 'cond_mode': 'guide', 'cond_strength': 0.70, 'image_args': [], 'chain_from': 'bus_stop_9x16_01_approach',
        'prompt': (
            'Vertical 9:16 cinematic 35mm continuation with the exact same middle-aged man and pretty woman at the bus stop from the starting frame. '
            'The pretty woman walks away down the sidewalk. '
            'The man turns to look directly at the camera with a wry smile and says: "At least she smiled." '
            'Light city ambience and subtle upbeat music sting.'
        ),
    },
]

WARREN_LINES = [
    ('warren_16nov_01_toolbox', 193, "It doesn't matter whether you use pickup lines or start conversations based on the situation."),
    ('warren_16nov_02_respectful', 193, "You should have both skills in your toolbox and use them as needed. Just keep it respectful."),
    ('warren_16nov_03_nightclub', 241, "When you're in a busy nightclub or bar, a pickup line might be needed because you can't think of anything to say and you're nervous."),
    ('warren_16nov_04_coffeeshop', 241, "When you're out at the coffee shop or on the train, a natural conversation based on what's happening around you might be appropriate."),
    ('warren_16nov_05_say_something', 193, "It really depends. The most important thing is that you say something. You will have no chance if you stay silent."),
]
for name, nf, line in WARREN_LINES:
    JOBS.append({
        'name': name, 'group': 'coach_warren_9x16_slow', 'character': 'coach_warren',
        'aspect': '9:16', 'width': W9, 'height': H9, 'num_frames': nf, 'seed': rand_seed(),
        'word_count': len(line.split()), 'cond_mode': 'guide', 'cond_strength': 0.75,
        'image_args': ['--image', '/content/coach_warren_ref_9x16.jpg', '0', '0.75', 'guide'], 'chain_from': None,
        'prompt': f'{WARREN_9X16_SCENE}He says slowly and calmly: "{line}" {WARREN_AUDIO}',
    })

COMMON_ARGS = [
    'uv', 'run', '--no-sync', 'python', '-m', 'ltx_pipelines.distilled',
    '--transformer-path', '/content/models/ltx-2.5/diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    '--text-encoder-path', '/content/models/ltx-2.5/text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    '--video-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-video-vae-bf16.safetensors',
    '--audio-vae-path', '/content/models/ltx-2.5/vae/ltx-2.5-audio-vae-bf16.safetensors',
    '--duration-head-path', '/content/models/ltx-2.5/model_patches/ltx-2.5-duration-head-bf16.safetensors',
    '--spatial-upsampler-path', '/content/models/ltx-2.5/latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
]

def sample_gpu():
    out = subprocess.check_output([
        'nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.free,temperature.gpu,power.draw',
        '--format=csv,noheader,nounits'
    ], text=True).strip().split(',')
    return [float(x.strip()) for x in out]

def polish_audio(src_mp4, dst_mp4):
    subprocess.run([
        'ffmpeg', '-y', '-i', src_mp4, '-c:v', 'copy',
        '-af', 'highpass=f=80,acompressor=threshold=-18dB:ratio=3,loudnorm=I=-16:TP=-1.5:LRA=11',
        '-c:a', 'aac', '-b:a', '192k', dst_mp4
    ], check=False, capture_output=True)

telem_csv_path = f'{LOG_DIR}/gpu_telemetry.csv'
with open(telem_csv_path, 'w', newline='') as tf:
    csv.writer(tf).writerow(['timestamp', 'clip_name', 'gpu_util_pct', 'vram_used_mib', 'vram_free_mib', 'temp_c', 'power_w'])

jsonl_path = f'{LOG_DIR}/clips_dataset.jsonl'
summary = {'run_id': RUN_ID, 'gpu_name': GPU_NAME, 'gpu_vram_mib': GPU_VRAM_MIB, 'clips': []}
generated_paths = {}

for idx, job in enumerate(JOBS, 1):
    name, seed = job['name'], job['seed']
    w, h, nf = job['width'], job['height'], job['num_frames']
    img_args = list(job['image_args'])
    if job['chain_from']:
        prev_mp4 = generated_paths.get(job['chain_from'])
        if prev_mp4 and os.path.exists(prev_mp4):
            chain_img = f'/content/{job["chain_from"]}_last.jpg'
            subprocess.run(['ffmpeg', '-y', '-sseof', '-0.1', '-i', prev_mp4, '-update', '1', '-frames:v', '1', chain_img], check=True, capture_output=True)
            img_args = ['--image', chain_img, '0', str(job['cond_strength']), 'guide']

    out_filename = f'{name}_seed{seed}.mp4'
    local_out = f'/content/{out_filename}'
    drive_video = f'{VIDEO_DIR}/{out_filename}'
    drive_polished = f'{POLISHED_DIR}/{out_filename}'
    drive_log = f'{LOG_DIR}/{name}.log'
    cmd = COMMON_ARGS + [
        '--prompt', job['prompt'],
        '--height', str(h), '--width', str(w),
        '--num-frames', str(nf), '--seed', str(seed),
        '--output-path', local_out,
    ] + EXTRA_ARGS + img_args

    wps = round(job['word_count'] / (nf / 24.0), 2)
    print(f"\n[{idx}/{len(JOBS)}] {name} | {w}x{h} ({job['aspect']}) | {nf}f ({wps} w/s) | seed={seed}")
    stop_ev = threading.Event()
    samples = []
    def _poll():
        while not stop_ev.is_set():
            try:
                samples.append((time.time(), *sample_gpu()))
            except Exception:
                pass
            stop_ev.wait(2.0)
    pt = threading.Thread(target=_poll, daemon=True)
    t0 = time.time()
    pt.start()
    proc = subprocess.run(cmd, cwd='/content/LTX-2', env=SUBPROC_ENV, capture_output=True, text=True)
    stop_ev.set()
    pt.join(timeout=5)
    elapsed = time.time() - t0

    full_log = proc.stdout + '\n--- STDERR ---\n' + proc.stderr
    with open(drive_log, 'w') as lf:
        lf.write(full_log)
    with open(telem_csv_path, 'a', newline='') as tf:
        wr = csv.writer(tf)
        for ts, u, used, free, temp, pwr in samples:
            wr.writerow([round(ts, 3), name, u, used, free, temp, pwr])

    if proc.returncode != 0 or not os.path.exists(local_out):
        print(f'FAILED {name} (rc={proc.returncode}); see {drive_log}\n' + '\n'.join(full_log.splitlines()[-20:]))
        continue

    generated_paths[name] = local_out
    shutil.copy2(local_out, drive_video)
    polish_audio(drive_video, drive_polished)
    probe = json.loads(subprocess.check_output([
        'ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', drive_video
    ], text=True))
    vs = next((s for s in probe['streams'] if s['codec_type'] == 'video'), {})
    aus = next((s for s in probe['streams'] if s['codec_type'] == 'audio'), {})
    peak_vram = max((s[2] for s in samples), default=0.0)
    mean_util = sum(s[1] for s in samples)/len(samples) if samples else 0.0
    mean_pwr = sum(s[5] for s in samples)/len(samples) if samples else 0.0
    max_temp = max((s[4] for s in samples), default=0.0)
    energy_wh = (mean_pwr * elapsed) / 3600.0
    est_runpod_usd = (elapsed / 3600.0) * RUNPOD_A100_HOURLY_USD
    est_colab_cu = (elapsed / 3600.0) * COLAB_A100_UNITS_PER_HR
    natten_fallback = 'Falling back to' in full_log

    ml_row = {
        'run_id': RUN_ID, 'gpu_name': GPU_NAME, 'gpu_vram_mib': GPU_VRAM_MIB,
        'clip_name': name, 'group': job['group'], 'character': job['character'],
        'aspect': job['aspect'], 'seed': seed, 'width': w, 'height': h, 'num_frames': nf,
        'word_count': job['word_count'], 'target_words_per_sec': wps,
        'cond_mode': job['cond_mode'], 'cond_strength': job['cond_strength'],
        'elapsed_s': round(elapsed, 2), 'sec_per_frame': round(elapsed / nf, 4),
        'peak_vram_mib': round(peak_vram, 1), 'mean_gpu_util_pct': round(mean_util, 1),
        'mean_power_w': round(mean_pwr, 1), 'max_temp_c': round(max_temp, 1), 'energy_wh': round(energy_wh, 3),
        'est_runpod_a100_usd': round(est_runpod_usd, 4), 'est_colab_compute_units': round(est_colab_cu, 3),
        'video_bitrate_bps': int(vs.get('bit_rate', 0)), 'audio_bitrate_bps': int(aus.get('bit_rate', 0)),
        'file_size_bytes': int(probe['format']['size']), 'natten_active': not natten_fallback,
    }
    with open(jsonl_path, 'a') as jf:
        jf.write(json.dumps(ml_row) + '\n')
    summary['clips'].append({**ml_row, 'video_path': drive_video, 'log_path': drive_log, 'ffprobe': probe})
    print(f"Saved {out_filename} in {elapsed:.1f}s | v_bitrate={ml_row['video_bitrate_bps']//1000}kbps | peak_vram={peak_vram:.0f}MiB | ~${est_runpod_usd:.4f}")

tot_infer_s = sum(c['elapsed_s'] for c in summary['clips'])
tot_wall_s = time.time() - RUN_START_TS
summary['totals'] = {
    'num_clips': len(summary['clips']),
    'total_inference_s': round(tot_infer_s, 2),
    'total_wall_s': round(tot_wall_s, 2),
    'total_energy_wh': round(sum(c['energy_wh'] for c in summary['clips']), 3),
    'est_runpod_a100_usd_infer': round((tot_infer_s / 3600.0) * RUNPOD_A100_HOURLY_USD, 4),
    'est_colab_cu_wall': round((tot_wall_s / 3600.0) * COLAB_A100_UNITS_PER_HR, 3),
}
with open(f'{LOG_DIR}/run_summary.json', 'w') as sf:
    json.dump(summary, sf, indent=2)
print('\nTotals:', json.dumps(summary['totals'], indent=2))


In [ ]:
#@title 8. Zip All Videos, Telemetry & Logs to Google Drive
import os, zipfile

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RUN_DIR):
        for fname in sorted(files):
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, os.path.dirname(RUN_DIR))
            zf.write(full_path, arcname=rel_path)
os.sync()
print(f'Zipped all artifacts ({os.path.getsize(ZIP_PATH)/(1024*1024):.2f} MB) -> {ZIP_PATH}')

In [ ]:
#@title 9. Flush Drive & Auto-Shutdown (or Pause for Local Browser Download)
from google.colab import runtime

if BASE_DRIVE_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed and unmounted. Unassigning and deleting Colab runtime...')
    runtime.unassign()
else:
    from google.colab import files
    print(f'Drive not mounted. Triggering browser download for {ZIP_PATH} ...')
    files.download(ZIP_PATH)
    input('Press Enter AFTER your browser download completes to delete the Colab runtime (or disconnect manually)...')
    runtime.unassign()
